In [34]:
import pandas as pd
import numpy as np

df=pd.read_csv('/content/dataset_limpio_SIMA (1).csv')

df['Estación'].unique()

df['Fecha']

,Fecha
0,2020-01-01 00:00:00
1,2020-01-01 01:00:00
2,2020-01-01 02:00:00
3,2020-01-01 03:00:00
4,2020-01-01 04:00:00
...,...
754598,2025-12-31 19:00:00
754599,2025-12-31 20:00:00
754600,2025-12-31 21:00:00
754601,2025-12-31 22:00:00


In [35]:
import pandas as pd

comparar=df.copy()
comparar['Fecha'] = pd.to_datetime(comparar['Fecha'])
fechas_ordenadas = comparar['Fecha'].dropna().sort_values()

# 2. Calculamos las diferencias entre filas consecutivas
diferencias = fechas_ordenadas.diff()

# 3. Filtramos para eliminar las diferencias de 0 segundos (mismo timestamp)
diferencias_reales = diferencias[diferencias > pd.Timedelta(0)]

# 4. Obtenemos el intervalo más común y el desglose de frecuencias
frecuencia_mas_comun = diferencias_reales.mode()[0]
minutos = frecuencia_mas_comun.total_seconds() / 60

print(f"Frecuencia más común (filtrando duplicados): Cada {frecuencia_mas_comun}")
print(f"Es decir, aparece un nuevo registro cada {int(minutos)} minutos.\n")

print("=== Top 5 intervalos de tiempo más frecuentes ===")
print(diferencias_reales.value_counts().head(5))

Frecuencia más común (filtrando duplicados): Cada 0 days 01:00:00
Es decir, aparece un nuevo registro cada 60 minutos.

=== Top 5 intervalos de tiempo más frecuentes ===
Fecha
0 days 01:00:00    52607
Name: count, dtype: int64


In [36]:
import pandas as pd
import numpy as np

df['Fecha'] = pd.to_datetime(
    df['Fecha']
        .astype(str)
        .str.replace('a. m.', 'AM', regex=False)
        .str.replace('p. m.', 'PM', regex=False),
    format='mixed',
    dayfirst=True
)

# 2. Quitar duplicados exactos de (Estación, Fecha) y ordenar cronológicamente
df = df.drop_duplicates(subset=['Estación', 'Fecha'])
df = df.sort_values(['Estación', 'Fecha']).reset_index(drop=True)

In [37]:
# ========================OPCIÓN 2=============================


# HALFLIFE = '3h'  # a las 3h, el peso de una observación ya cayó a la mitad

# df_b = df.set_index('Fecha')

# resultado_b = (
#     df_b.groupby('Estación')['PM10']
#         .apply(lambda g: g.ewm(halflife=HALFLIFE, times=g.index).mean())
#         .reset_index()
#         .rename(columns={'PM10': 'PM10_ewma'})
# )

# df = df.merge(
#     resultado_b[['Estación', 'Fecha', 'PM10_ewma']],
#     on=['Estación', 'Fecha'],
#     how='left'
# )

In [38]:
MIN_LECTURAS_A = 10  # de las 12 esperadas en 12h a cada hora (~83% cobertura)

def media_ponderada_lineal(valores):
    """Pesos crecientes: la última observación de la ventana pesa más.
    Si solo hay NaN en la ventana, devuelve NaN en vez de fallar."""
    valores = np.asarray(valores, dtype=float)
    mask = ~np.isnan(valores)
    if mask.sum() == 0:
        return np.nan
    pesos = np.arange(1, len(valores) + 1)[mask]
    return np.average(valores[mask], weights=pesos)

df_a = df.set_index('Fecha')

resultado_a = (
    df_a.groupby('Estación')['PM10']
        .rolling(window='12h', min_periods=1)
        .agg(['count'])  # para poder invalidar por baja cobertura
)
resultado_a = resultado_a.reset_index()

pesos_a = (
    df_a.groupby('Estación')['PM10']
        .rolling(window='12h', min_periods=1)
        .apply(media_ponderada_lineal, raw=True)
        .reset_index()
        .rename(columns={'PM10': 'PM10_wma12h'})
)

resultado_a = resultado_a.merge(pesos_a, on=['Estación', 'Fecha'])
resultado_a.loc[resultado_a['count'] < MIN_LECTURAS_A, 'PM10_wma12h'] = pd.NA

df = df.merge(
    resultado_a[['Estación', 'Fecha', 'PM10_wma12h', 'count']],
    on=['Estación', 'Fecha'],
    how='left'
).rename(columns={'count': 'PM10_wma12h_n'})



In [39]:
df.shape

(754603, 20)

In [40]:
df['PM10_wma12h'].isna().sum()

np.int64(135)

In [41]:
df.tail()

,Fecha,CO,NO,NO2,NOX,O3,PM10,PM2.5,BP,RAINF,RH,SO2,SR,TEMP,WS,WD,Año,Estación,PM10_wma12h,PM10_wma12h_n
754598,2025-12-31 19:00:00,0.93,2.5,13.3,15.7,41.0,19.0,17.0,717.1,0.0,45.0,2.8,0.002,15.77,4.8,213.0,2025,SUR,27.782051,12.0
754599,2025-12-31 20:00:00,1.07,3.2,21.5,24.5,29.0,23.0,8.0,717.2,0.0,49.0,2.8,0.001,14.84,5.2,132.0,2025,SUR,27.051282,12.0
754600,2025-12-31 21:00:00,1.16,4.3,24.1,28.3,23.0,40.0,28.0,717.2,0.0,53.0,2.2,0.001,13.82,7.3,141.0,2025,SUR,28.846154,12.0
754601,2025-12-31 22:00:00,1.16,3.4,19.8,23.1,23.0,39.0,25.0,717.1,0.0,56.0,2.2,0.001,13.03,6.8,144.0,2025,SUR,30.243590,12.0
754602,2025-12-31 23:00:00,1.11,2.5,15.8,18.2,22.0,34.0,26.0,717.1,0.0,60.0,2.2,0.001,11.97,3.0,234.0,2025,SUR,30.717949,12.0


In [42]:
print(df['PM10_wma12h'].nlargest(10))


87662     738.448718
87663     731.935897
87661     709.179487
87664     697.705128
87660     637.666667
87665     632.217949
87666     559.525641
686615    549.337754
87659     541.192308
368904    513.576923
Name: PM10_wma12h, dtype: float64


In [43]:


# =====================================================================
# PASO 2: Condición >= 90, y detección de "runs" continuos reales
# =====================================================================

df = df.sort_values(['Estación', 'Fecha']).reset_index(drop=True)

UMBRAL = 90
df['cond90'] = df['PM10_wma12h'] >= UMBRAL

gap_h = df.groupby('Estación')['Fecha'].diff().dt.total_seconds() / 3600
es_continuo = (gap_h == 1)
cambio_estacion = df['Estación'] != df['Estación'].shift()
cambio_cond = df['cond90'] != df['cond90'].shift()
run_break = cambio_estacion | cambio_cond | (~es_continuo)
df['run_id'] = run_break.cumsum()

run_len = df.groupby('run_id')['cond90'].transform('size')
pos_en_run = df.groupby('run_id').cumcount()

# =====================================================================
# PASO 3: Encodeo "vanilla" (sin periodo de silencio todavía)
# =====================================================================

codigo_raw = np.zeros(len(df), dtype=int)
mask_true = df['cond90'].values
L = run_len.values
pos = pos_en_run.values

codigo_raw[mask_true & (L == 1) & (pos == 0)] = 2
codigo_raw[mask_true & (L == 2) & (pos == 1)] = 2
codigo_raw[mask_true & (L >= 3) & (pos == 2)] = 1

# =====================================================================
# PASO 4: Periodo de silencio de 12h REALES tras cada alerta (código 1)
#          -> cualquier código dentro de esas 12h se fuerza a 0
#          -> después de la hora 12, la lógica se reinicia normalmente
# =====================================================================

codigo_final = codigo_raw.copy()
fechas = df['Fecha'].to_numpy()
estaciones = df['Estación'].to_numpy()

ultimo_disparo = {}  # {estación: timestamp del último código 1 válido}

for j in range(len(df)):
    est = estaciones[j]
    f = fechas[j]

    ud = ultimo_disparo.get(est)
    if ud is not None:
        horas_desde = (f - ud) / np.timedelta64(1, 'h')
        if 0 < horas_desde <= 24:  #VENTANAS CASO 1
            codigo_final[j] = 0
            continue  # dentro del silencio: no cuenta como nuevo disparo

    if codigo_raw[j] == 1:
        ultimo_disparo[est] = f

df['PM10_alerta_code'] = codigo_final


In [44]:
df.tail()

,Fecha,CO,NO,NO2,NOX,O3,PM10,PM2.5,BP,RAINF,...,TEMP,WS,WD,Año,Estación,PM10_wma12h,PM10_wma12h_n,cond90,run_id,PM10_alerta_code
754598,2025-12-31 19:00:00,0.93,2.5,13.3,15.7,41.0,19.0,17.0,717.1,0.0,...,15.77,4.8,213.0,2025,SUR,27.782051,12.0,False,15675,0
754599,2025-12-31 20:00:00,1.07,3.2,21.5,24.5,29.0,23.0,8.0,717.2,0.0,...,14.84,5.2,132.0,2025,SUR,27.051282,12.0,False,15675,0
754600,2025-12-31 21:00:00,1.16,4.3,24.1,28.3,23.0,40.0,28.0,717.2,0.0,...,13.82,7.3,141.0,2025,SUR,28.846154,12.0,False,15675,0
754601,2025-12-31 22:00:00,1.16,3.4,19.8,23.1,23.0,39.0,25.0,717.1,0.0,...,13.03,6.8,144.0,2025,SUR,30.243590,12.0,False,15675,0
754602,2025-12-31 23:00:00,1.11,2.5,15.8,18.2,22.0,34.0,26.0,717.1,0.0,...,11.97,3.0,234.0,2025,SUR,30.717949,12.0,False,15675,0


In [45]:
df['PM10_alerta_code'].nunique()

3

In [46]:
df['PM10_alerta_code'].value_counts()

,count
PM10_alerta_code,
0,748733
1,5125
2,745


In [47]:
# from google.colab import files

# df.to_csv("dataset_24hrs.csv",index=False, encoding='Utf-8-sig')


In [48]:
# files.download("dataset_24hrs.csv")

# Modelaje

In [49]:
!pip install linearmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 8.2 MB/s eta 0:00:00


In [50]:
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS

# =====================================================================
# PASO 0: Construir el DataFrame `alertas` (Estación, Fecha_inicio,
#          Fecha_fin) directamente desde PM10_alerta_code.
#
#   Cada vez que aparece un código 1 (el disparo real de la alerta),
#   se considera que la alerta está activa durante las siguientes 24h
#   exactas -- consistente con el periodo de silencio de 24h que ya
#   usamos para evitar que un mismo episodio dispare varias alertas.
# =====================================================================

DURACION_ALERTA = pd.Timedelta(hours=24)

disparos = df.loc[df['PM10_alerta_code'] == 1, ['Estación', 'Fecha']].copy()
disparos = disparos.rename(columns={'Fecha': 'Fecha_inicio'})
disparos['Fecha_fin'] = disparos['Fecha_inicio'] + DURACION_ALERTA

alertas = disparos.reset_index(drop=True)

print(f"Total de alertas detectadas: {len(alertas)}")
print(alertas)

# =====================================================================
# PASO 1: Construir la variable "Periodo" (categórica) a partir de
#          `alertas` (inicio/fin de cada alerta)
# =====================================================================

VENTANA = pd.Timedelta(hours=72)

df['Periodo'] = 'Base'

for _, row in alertas.iterrows():
    est = row['Estación']
    inicio, fin = row['Fecha_inicio'], row['Fecha_fin']

    m_est = df['Estación'] == est
    m_pre = m_est & (df['Fecha'] >= inicio - VENTANA) & (df['Fecha'] < inicio)
    m_dur = m_est & (df['Fecha'] >= inicio) & (df['Fecha'] <= fin)
    m_post = m_est & (df['Fecha'] > fin) & (df['Fecha'] <= fin + VENTANA)

    df.loc[m_pre, 'Periodo'] = 'Pre_alerta'
    df.loc[m_dur, 'Periodo'] = 'Durante_alerta'
    df.loc[m_post, 'Periodo'] = 'Post_alerta'

df['Periodo'] = pd.Categorical(
    df['Periodo'],
    categories=['Base', 'Pre_alerta', 'Durante_alerta', 'Post_alerta']
)

print("\nDistribución de Periodo:")
print(df['Periodo'].value_counts())

# =====================================================================
# PASO 2: Descomponer WD (dirección del viento) en seno/coseno
# =====================================================================

df['WD_sin'] = np.sin(np.radians(df['WD']))
df['WD_cos'] = np.cos(np.radians(df['WD']))

# =====================================================================
# PASO 3: Controles cíclicos (hora del día, día de la semana)
# =====================================================================

df['Hora'] = df['Fecha'].dt.hour.astype('category')
df['DiaSemana'] = df['Fecha'].dt.dayofweek.astype('category')

# =====================================================================
# PASO 4: Variable dependiente (nivel y log) + estructura de panel
# =====================================================================

df['PM10_log'] = np.log(df['PM10'].clip(lower=0.1))

panel = df.dropna(
    subset=['PM10', 'Periodo', 'TEMP', 'RH', 'WS', 'WD_sin', 'WD_cos']
).copy()

panel = panel.set_index(['Estación', 'Fecha'])

# =====================================================================
# PASO 5: Estimación con efectos fijos de Estación + controles
#          cíclicos, errores estándar clustered por Estación
# =====================================================================

controles = ['TEMP', 'RH', 'WS', 'WD_sin', 'WD_cos']

def correr_modelo(y_var):
    modelo = PanelOLS.from_formula(
        f"{y_var} ~ 1 + Periodo + {' + '.join(controles)} + "
        f"C(Hora) + C(DiaSemana) + EntityEffects",
        data=panel
    )
    return modelo.fit(cov_type='clustered', cluster_entity=True)

resultado_nivel = correr_modelo('PM10')
resultado_log = correr_modelo('PM10_log')

print("\n" + "="*70)
print("MODELO EN NIVEL")
print("="*70)
print(resultado_nivel.summary)

print("\n" + "="*70)
print("MODELO EN LOGARITMO")
print("="*70)
print(resultado_log.summary)

Total de alertas detectadas: 5125
     Estación        Fecha_inicio           Fecha_fin
0          CE 2020-01-01 11:00:00 2020-01-02 11:00:00
1          CE 2020-01-06 16:00:00 2020-01-07 16:00:00
2          CE 2020-01-09 07:00:00 2020-01-10 07:00:00
3          CE 2020-01-13 02:00:00 2020-01-14 02:00:00
4          CE 2020-01-22 23:00:00 2020-01-23 23:00:00
...       ...                 ...                 ...
5120      SUR 2025-03-15 18:00:00 2025-03-16 18:00:00
5121      SUR 2025-03-19 11:00:00 2025-03-20 11:00:00
5122      SUR 2025-07-04 15:00:00 2025-07-05 15:00:00
5123      SUR 2025-11-08 07:00:00 2025-11-09 07:00:00
5124      SUR 2025-11-09 14:00:00 2025-11-10 14:00:00

[5125 rows x 3 columns]

Distribución de Periodo:
Periodo
Base              318951
Pre_alerta        291878
Post_alerta        91932
Durante_alerta     51842
Name: count, dtype: int64

MODELO EN NIVEL
                          PanelOLS Estimation Summary                           
Dep. Variable:                   PM